In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/merged/panel_with_controls_1992_2024.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115640 entries, 0 to 115639
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   sender_iso3          115640 non-null  object 
 1   recipient_iso3       115640 non-null  object 
 2   year                 115640 non-null  int64  
 3   arms_tiv             115640 non-null  float64
 4   bilateral_oda        103839 non-null  float64
 5   bilateral_debt       18868 non-null   float64
 6   colonial_tie         115640 non-null  int64  
 7   journalist_killings  115640 non-null  float64
 8   gdp_per_capita       112889 non-null  float64
 9   gdp_per_capita_log   112889 non-null  float64
 10  population           114907 non-null  float64
 11  population_log       114907 non-null  float64
 12  v2x_polyarchy        109836 non-null  float64
 13  armed_conflict       114996 non-null  float64
 14  conflict_intensity   114996 non-null  float64
dtypes: float64(11), i

In [5]:
df2 = pd.read_csv('../data/processed/trade_dependency_engineering/p4_trade_engineering.csv')

df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415690 entries, 0 to 415689
Data columns (total 16 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   year                        415690 non-null  float64
 1   country_a                   415690 non-null  object 
 2   a_iso3                      415690 non-null  object 
 3   a_gdp                       415690 non-null  float64
 4   country_b                   415690 non-null  object 
 5   b_iso3                      415690 non-null  object 
 6   b_gdp                       415690 non-null  float64
 7   total_trade_relation_value  415690 non-null  float64
 8   a_dependency_pct            415690 non-null  float64
 9   b_dependency_pct            415690 non-null  float64
 10  a_ic                        415690 non-null  int64  
 11  b_ic                        415690 non-null  int64  
 12  a_total_trade_all_partners  415690 non-null  float64
 13  b_total_trade_

In [6]:
import pandas as pd
import numpy as np

def analyze_dataframe(df):
    """
    Comprehensive DataFrame analysis showing missing values, unique values, and value counts
    """
    
    print("=" * 80)
    print("DATAFRAME ANALYSIS")
    print("=" * 80)
    
    # Section 1: Basic Info
    print("\n1. BASIC DATAFRAME INFORMATION")
    print("-" * 40)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Total cells: {df.size}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Section 2: Missing Values Analysis
    print("\n2. MISSING VALUES BY COLUMN")
    print("-" * 40)
    
    # Create a DataFrame with missing value info
    missing_info = pd.DataFrame({
        'Column': df.columns,
        'Data Type': [df[col].dtype for col in df.columns],
        'Non-Null Count': [df[col].count() for col in df.columns],
        'Missing Count': [df[col].isnull().sum() for col in df.columns],
        'Missing %': [round(df[col].isnull().sum() / len(df) * 100, 2) for col in df.columns]
    })
    
    # Sort by missing count descending
    missing_info = missing_info.sort_values('Missing Count', ascending=False)
    
    # Format the output
    for _, row in missing_info.iterrows():
        missing_bar = '█' * int(row['Missing %'] / 5)  # Visual bar for missing percentage
        print(f"{row['Column'][:30]:<30} "
              f"| Type: {str(row['Data Type']):<10} "
              f"| Missing: {int(row['Missing Count']):>6} "
              f"| {row['Missing %']:>5}% {missing_bar}")
    
    # Section 3: Unique Values Analysis (for object/category columns)
    print("\n3. UNIQUE VALUES IN CATEGORICAL COLUMNS")
    print("-" * 40)
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    
    if len(categorical_cols) > 0:
        unique_info = pd.DataFrame({
            'Column': categorical_cols,
            'Unique Values': [df[col].nunique() for col in categorical_cols],
            'Sample Values': [str(list(df[col].dropna().unique()[:5])) for col in categorical_cols]
        })
        unique_info = unique_info.sort_values('Unique Values', ascending=False)
        
        for _, row in unique_info.iterrows():
            print(f"{row['Column'][:30]:<30} "
                  f"| Unique: {row['Unique Values']:<5} "
                  f"| Samples: {row['Sample Values'][:50]}")
    else:
        print("No categorical columns found")
    
    # Section 4: Numeric Columns Statistics
    print("\n4. NUMERIC COLUMNS SUMMARY")
    print("-" * 40)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 0:
        numeric_stats = df[numeric_cols].describe().T
        numeric_stats['missing'] = df[numeric_cols].isnull().sum()
        print(numeric_stats[['count', 'missing', 'mean', 'min', 'max']])
    else:
        print("No numeric columns found")
    
    return missing_info

def show_value_counts(df, column_name, top_n=10):
    """
    Display value counts for a specific column with formatting
    """
    print("\n" + "=" * 80)
    print(f"VALUE COUNTS FOR COLUMN: '{column_name}'")
    print("=" * 80)
    
    if column_name not in df.columns:
        print(f"Error: Column '{column_name}' not found in DataFrame")
        return
    
    # Get value counts
    counts = df[column_name].value_counts()
    
    # Get value counts with percentages
    counts_pct = df[column_name].value_counts(normalize=True) * 100
    
    # Create a combined DataFrame
    value_counts_df = pd.DataFrame({
        'Count': counts,
        'Percentage': counts_pct,
        'Cumulative %': counts_pct.cumsum()
    })
    
    # Show top N or all if less than top_n
    display_count = min(top_n, len(counts))
    
    print(f"\nTop {display_count} values (out of {len(counts)} unique values):")
    print("-" * 60)
    
    for i, (value, count) in enumerate(counts.head(display_count).items(), 1):
        percentage = counts_pct[value]
        bar = '█' * int(percentage / 2)  # Visual bar (max 50 chars)
        
        # Truncate value if too long
        value_str = str(value)[:30]
        
        print(f"{i:2}. {value_str:<30} "
              f"| Count: {count:<8} "
              f"| {percentage:>5.1f}% {bar}")
    
    # Show missing values if any
    missing_count = df[column_name].isnull().sum()
    if missing_count > 0:
        missing_pct = (missing_count / len(df)) * 100
        print(f"\nMissing values: {missing_count} ({missing_pct:.1f}%)")
    
    return value_counts_df

In [8]:
analyze_dataframe(df)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 115640 rows × 15 columns
Total cells: 1734600
Memory usage: 22.94 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
bilateral_debt                 | Type: float64    | Missing:  96772 | 83.68% ████████████████
bilateral_oda                  | Type: float64    | Missing:  11801 |  10.2% ██
v2x_polyarchy                  | Type: float64    | Missing:   5804 |  5.02% █
gdp_per_capita                 | Type: float64    | Missing:   2751 |  2.38% 
gdp_per_capita_log             | Type: float64    | Missing:   2751 |  2.38% 
population                     | Type: float64    | Missing:    733 |  0.63% 
population_log                 | Type: float64    | Missing:    733 |  0.63% 
conflict_intensity             | Type: float64    | Missing:    644 |  0.56% 
armed_conflict                 | Type: float64    | Missing:    644 |  0.56% 
colonial_tie                   | Type: in

,Column,Data Type,Non-Null Count,Missing Count,Missing %
5,bilateral_debt,float64,18868,96772,83.68
4,bilateral_oda,float64,103839,11801,10.20
12,v2x_polyarchy,float64,109836,5804,5.02
8,gdp_per_capita,float64,112889,2751,2.38
9,gdp_per_capita_log,float64,112889,2751,2.38
10,population,float64,114907,733,0.63
11,population_log,float64,114907,733,0.63
14,conflict_intensity,float64,114996,644,0.56
13,armed_conflict,float64,114996,644,0.56
6,colonial_tie,int64,115640,0,0.00


In [7]:
analyze_dataframe(df2)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 415690 rows × 16 columns
Total cells: 6651040
Memory usage: 128.32 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
year                           | Type: float64    | Missing:      0 |   0.0% 
country_a                      | Type: object     | Missing:      0 |   0.0% 
a_iso3                         | Type: object     | Missing:      0 |   0.0% 
a_gdp                          | Type: float64    | Missing:      0 |   0.0% 
country_b                      | Type: object     | Missing:      0 |   0.0% 
b_iso3                         | Type: object     | Missing:      0 |   0.0% 
b_gdp                          | Type: float64    | Missing:      0 |   0.0% 
total_trade_relation_value     | Type: float64    | Missing:      0 |   0.0% 
a_dependency_pct               | Type: float64    | Missing:      0 |   0.0% 
b_dependency_pct               | Type: float64    | Missing

,Column,Data Type,Non-Null Count,Missing Count,Missing %
0,year,float64,415690,0,0.0
1,country_a,object,415690,0,0.0
2,a_iso3,object,415690,0,0.0
3,a_gdp,float64,415690,0,0.0
4,country_b,object,415690,0,0.0
5,b_iso3,object,415690,0,0.0
6,b_gdp,float64,415690,0,0.0
7,total_trade_relation_value,float64,415690,0,0.0
8,a_dependency_pct,float64,415690,0,0.0
9,b_dependency_pct,float64,415690,0,0.0


In [9]:
df2.head()

,year,country_a,a_iso3,a_gdp,country_b,b_iso3,b_gdp,total_trade_relation_value,a_dependency_pct,b_dependency_pct,a_ic,b_ic,a_total_trade_all_partners,b_total_trade_all_partners,a_trade_share,b_trade_share
0,2012.0,Barbados,BRB,20804.115000,"South Sudan, Republic of",SSD,1109.260501,0.000024,1.153618e-07,2.163604e-06,4,1,1825.094587,232.631092,1.315000e-08,1.031676e-07
1,2021.0,Côte d'Ivoire,CIV,2455.981276,"Micronesia, Federated States of",FSM,3508.223142,0.000025,1.017923e-06,7.126115e-07,2,2,43604.470815,513.942559,5.733357e-10,4.864357e-08
2,2013.0,Denmark,DNK,61377.594059,Tuvalu,TUV,3510.216401,0.000035,5.702407e-08,9.970895e-07,4,3,205206.269437,108.816958,1.705601e-10,3.216410e-07
3,2016.0,"Latvia, Republic of",LVA,13838.526682,"São Tomé and Príncipe, Democratic Republic of",STP,1434.870181,0.000055,3.974412e-07,3.833099e-06,4,2,32298.999051,160.416332,1.702839e-09,3.428579e-07
4,2016.0,Gabon,GAB,6677.070039,Zambia,ZMB,1239.085279,0.000060,8.985977e-07,4.842282e-06,3,2,12070.608990,11552.920906,4.970752e-09,5.193492e-09


In [10]:
# Create a normalized pair identifier (sorted pair)
import pandas as pd

# Create temporary columns with sorted country names
df2['country_pair'] = df2.apply(
    lambda row: tuple(sorted([row['country_a'], row['country_b']])), 
    axis=1
)

# Check for duplicates in the normalized pairs
duplicate_pairs = df2[df2.duplicated(subset=['country_pair', 'year'], keep=False)]
print("Rows that are part of reciprocal pairs:")
print(duplicate_pairs[['year', 'country_a', 'country_b', 'country_pair']])

Rows that are part of reciprocal pairs:
          year             country_a                       country_b  \
1426    1993.0  Netherlands Antilles  St. Vincent and the Grenadines   
1427    1993.0  Netherlands Antilles  St. Vincent and the Grenadines   
13709   1993.0  Netherlands Antilles             St. Kitts and Nevis   
13710   1993.0  Netherlands Antilles             St. Kitts and Nevis   
15393   1993.0               Grenada            Netherlands Antilles   
...        ...                   ...                             ...   
412313  1993.0      Netherlands, The                Papua New Guinea   
412411  1992.0      Netherlands, The                Papua New Guinea   
412412  1992.0      Netherlands, The                Papua New Guinea   
414192  1993.0              Zimbabwe            Netherlands Antilles   
414193  1993.0              Zimbabwe            Netherlands Antilles   

                                             country_pair  
1426    (Netherlands Antilles, St. 